# Word Embeddings

Dense vector representations that capture semantic meaning.

## Learning Objectives

- Understand word embedding concepts
- Implement Skip-gram from scratch
- Use pre-trained embeddings
- Visualize word relationships

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

import torch
import torch.nn as nn
import torch.optim as optim

plt.style.use('seaborn-v0_8-whitegrid')
print("Libraries loaded!")

## 1. Why Word Embeddings?

| One-Hot | Embeddings |
|---------|------------|
| Sparse (mostly zeros) | Dense (all values meaningful) |
| High dimensional | Low dimensional (50-300) |
| No semantic info | Captures relationships |
| king = [0,0,1,0,...] | king ≈ [0.2, -0.5, 0.8, ...] |

In [ ]:
# Demo: One-hot vs Embeddings
vocab = ['king', 'queen', 'man', 'woman', 'apple', 'orange']

# One-hot (sparse, no relationships)
one_hot = np.eye(len(vocab))
print("One-Hot Encodings:")
for i, word in enumerate(vocab[:3]):
    print(f"  {word}: {one_hot[i].astype(int)}")

# Hypothetical embeddings (dense, semantic)
embeddings = np.array([
    [0.8, 0.9, 0.2],  # king: high royalty, high power
    [0.8, 0.3, 0.2],  # queen: high royalty, lower power
    [0.1, 0.9, 0.2],  # man: low royalty, high power
    [0.1, 0.3, 0.2],  # woman: low royalty, lower power
    [0.0, 0.0, 0.9],  # apple: fruit
    [0.0, 0.0, 0.8],  # orange: fruit
])

print("\nEmbeddings (3D):")
for i, word in enumerate(vocab):
    print(f"  {word}: {embeddings[i]}")

In [ ]:
# Semantic relationships via embeddings
print("=== Semantic Relationships ===")

# Cosine similarity
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

word_to_idx = {w: i for i, w in enumerate(vocab)}

pairs = [('king', 'queen'), ('king', 'man'), ('apple', 'orange'), ('king', 'apple')]
for w1, w2 in pairs:
    sim = cosine_sim(embeddings[word_to_idx[w1]], embeddings[word_to_idx[w2]])
    print(f"  {w1} <-> {w2}: {sim:.3f}")

In [ ]:
# Famous analogy: king - man + woman = queen
king = embeddings[word_to_idx['king']]
man = embeddings[word_to_idx['man']]
woman = embeddings[word_to_idx['woman']]
queen = embeddings[word_to_idx['queen']]

result = king - man + woman

print("king - man + woman = ?")
print(f"  Result vector: {result}")
print(f"  Queen vector:  {queen}")
print(f"  Similarity:    {cosine_sim(result, queen):.3f}")

## 2. Word2Vec: Skip-gram from Scratch

In [ ]:
# Training corpus
corpus = [
    "the cat sat on the mat",
    "the dog ran in the park",
    "the cat and dog played",
    "the bird flew over the house"
]

# Tokenize and build vocabulary
def tokenize(text):
    return text.lower().split()

all_words = []
for sentence in corpus:
    all_words.extend(tokenize(sentence))

word_counts = Counter(all_words)
vocab = sorted(set(all_words))
word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for i, w in enumerate(vocab)}

print(f"Vocabulary ({len(vocab)} words): {vocab}")
print(f"\nWord counts: {dict(word_counts)}")

In [ ]:
# Generate skip-gram pairs (center, context)
def generate_skipgrams(sentences, window_size=2):
    """Generate (center, context) pairs."""
    pairs = []
    for sentence in sentences:
        tokens = tokenize(sentence)
        for i, center in enumerate(tokens):
            # Context window
            start = max(0, i - window_size)
            end = min(len(tokens), i + window_size + 1)
            
            for j in range(start, end):
                if j != i:  # Skip center word itself
                    context = tokens[j]
                    pairs.append((word_to_idx[center], word_to_idx[context]))
    return pairs

skipgrams = generate_skipgrams(corpus, window_size=2)
print(f"Generated {len(skipgrams)} skip-gram pairs")
print(f"\nExamples:")
for center_idx, context_idx in skipgrams[:5]:
    print(f"  ({idx_to_word[center_idx]}, {idx_to_word[context_idx]})")

In [ ]:
# Skip-gram model in PyTorch
class SkipGram(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.center_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.context_embeddings = nn.Embedding(vocab_size, embedding_dim)
        
        # Initialize weights
        nn.init.xavier_uniform_(self.center_embeddings.weight)
        nn.init.xavier_uniform_(self.context_embeddings.weight)
    
    def forward(self, center, context):
        center_emb = self.center_embeddings(center)  # (batch, embed_dim)
        context_emb = self.context_embeddings(context)  # (batch, embed_dim)
        
        # Dot product
        scores = torch.sum(center_emb * context_emb, dim=1)
        return scores
    
    def get_embeddings(self):
        """Return word embeddings."""
        return self.center_embeddings.weight.detach().numpy()

# Initialize model
embedding_dim = 10
model = SkipGram(len(vocab), embedding_dim)
print(f"Model created with {embedding_dim}D embeddings")
print(f"Parameters: {sum(p.numel() for p in model.parameters())}")

In [ ]:
# Training loop
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.BCEWithLogitsLoss()

# Prepare data
centers = torch.tensor([p[0] for p in skipgrams])
contexts = torch.tensor([p[1] for p in skipgrams])
labels = torch.ones(len(skipgrams))  # Positive examples

# Add negative samples
n_negative = len(skipgrams) * 2
neg_centers = torch.randint(0, len(vocab), (n_negative,))
neg_contexts = torch.randint(0, len(vocab), (n_negative,))

all_centers = torch.cat([centers, neg_centers])
all_contexts = torch.cat([contexts, neg_contexts])
all_labels = torch.cat([labels, torch.zeros(n_negative)])

# Training
n_epochs = 100
losses = []

for epoch in range(n_epochs):
    optimizer.zero_grad()
    scores = model(all_centers, all_contexts)
    loss = criterion(scores, all_labels)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f"Training complete. Final loss: {losses[-1]:.4f}")

In [ ]:
# Visualize loss
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Skip-gram Training Loss')
plt.tight_layout()
plt.show()

In [ ]:
# Get learned embeddings
embeddings = model.get_embeddings()

# Check word similarities
print("=== Learned Word Similarities ===")
for word in ['cat', 'dog', 'the']:
    word_emb = embeddings[word_to_idx[word]].reshape(1, -1)
    sims = cosine_similarity(word_emb, embeddings).flatten()
    top_idx = sims.argsort()[-4:][::-1]  # Top 3 + self
    
    print(f"\n'{word}' most similar to:")
    for idx in top_idx:
        if idx != word_to_idx[word]:
            print(f"  {idx_to_word[idx]}: {sims[idx]:.3f}")

In [ ]:
# Visualize embeddings in 2D
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

plt.figure(figsize=(10, 8))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], s=100, alpha=0.7)

for i, word in enumerate(vocab):
    plt.annotate(
        word,
        (embeddings_2d[i, 0], embeddings_2d[i, 1]),
        fontsize=12,
        xytext=(5, 5),
        textcoords='offset points'
    )

plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title('Learned Word Embeddings (2D)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Using PyTorch Embedding Layers

In [ ]:
# Basic embedding layer
vocab_size = 1000
embed_dim = 64

embedding = nn.Embedding(vocab_size, embed_dim)

# Lookup single word
word_idx = torch.tensor([42])  # Word ID 42
word_vector = embedding(word_idx)
print(f"Word 42 embedding shape: {word_vector.shape}")
print(f"Embedding values: {word_vector[0, :5]}...")

In [ ]:
# Batch of sequences
batch_size = 4
seq_length = 10

# Random word indices
sequences = torch.randint(0, vocab_size, (batch_size, seq_length))
print(f"Input shape: {sequences.shape}")

# Embed all words
embedded = embedding(sequences)
print(f"Output shape: {embedded.shape}")
print(f"(batch_size, seq_length, embed_dim)")

In [ ]:
# Embedding layer with padding
embedding_with_pad = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embed_dim,
    padding_idx=0  # Index 0 will be zero vector
)

# Check padding
pad_vector = embedding_with_pad(torch.tensor([0]))
print(f"Padding vector (should be zeros): {pad_vector[0, :5]}")

## 4. Simple Text Classifier with Embeddings

In [ ]:
# Simple text classifier
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, n_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc = nn.Linear(embed_dim, n_classes)
    
    def forward(self, x):
        # x: (batch, seq_len)
        embedded = self.embedding(x)  # (batch, seq_len, embed_dim)
        
        # Mean pooling over sequence
        pooled = embedded.mean(dim=1)  # (batch, embed_dim)
        
        # Classify
        logits = self.fc(pooled)  # (batch, n_classes)
        return logits

# Create model
classifier = TextClassifier(
    vocab_size=5000,
    embed_dim=100,
    n_classes=3
)

# Forward pass
x = torch.randint(1, 5000, (8, 20))  # Batch of 8, seq length 20
logits = classifier(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {logits.shape}")

## 5. Word Embedding Arithmetic

In [ ]:
# Simulate pre-trained embeddings
np.random.seed(42)

# Vocabulary
words = [
    'king', 'queen', 'man', 'woman', 'prince', 'princess',
    'paris', 'france', 'berlin', 'germany', 'london', 'england',
    'walk', 'walking', 'run', 'running', 'swim', 'swimming'
]

# Create semantic clusters
embed_dim = 50
embeddings = np.random.randn(len(words), embed_dim) * 0.1

# Add semantic structure
royalty_vec = np.random.randn(embed_dim)
gender_vec = np.random.randn(embed_dim)
country_vec = np.random.randn(embed_dim)

# Royalty words
embeddings[0] += royalty_vec + gender_vec  # king
embeddings[1] += royalty_vec - gender_vec  # queen
embeddings[2] += gender_vec  # man
embeddings[3] += -gender_vec  # woman
embeddings[4] += royalty_vec * 0.8 + gender_vec  # prince
embeddings[5] += royalty_vec * 0.8 - gender_vec  # princess

word_to_idx = {w: i for i, w in enumerate(words)}

def analogy(a, b, c, embeddings, word_to_idx, words):
    """Solve: a is to b as c is to ?"""
    vec_a = embeddings[word_to_idx[a]]
    vec_b = embeddings[word_to_idx[b]]
    vec_c = embeddings[word_to_idx[c]]
    
    # Target: d = b - a + c
    target = vec_b - vec_a + vec_c
    
    # Find closest word
    similarities = cosine_similarity(target.reshape(1, -1), embeddings).flatten()
    
    # Exclude input words
    for word in [a, b, c]:
        similarities[word_to_idx[word]] = -1
    
    best_idx = similarities.argmax()
    return words[best_idx], similarities[best_idx]

# Test analogies
print("=== Word Analogies ===")
tests = [
    ('man', 'king', 'woman'),  # woman:queen
    ('king', 'prince', 'queen'),  # queen:princess
]

for a, b, c in tests:
    result, score = analogy(a, b, c, embeddings, word_to_idx, words)
    print(f"{a}:{b} :: {c}:? => {result} ({score:.3f})")

## 6. Embedding Aggregation Strategies

In [ ]:
# Different ways to combine word embeddings
embed_dim = 100
seq_length = 5

# Simulated embeddings for a sentence
sentence_embeddings = torch.randn(seq_length, embed_dim)

# 1. Mean pooling
mean_pool = sentence_embeddings.mean(dim=0)
print(f"Mean pooling shape: {mean_pool.shape}")

# 2. Max pooling
max_pool = sentence_embeddings.max(dim=0).values
print(f"Max pooling shape: {max_pool.shape}")

# 3. Sum pooling
sum_pool = sentence_embeddings.sum(dim=0)
print(f"Sum pooling shape: {sum_pool.shape}")

# 4. Concatenation (first + last)
concat_pool = torch.cat([sentence_embeddings[0], sentence_embeddings[-1]])
print(f"Concat pooling shape: {concat_pool.shape}")

In [ ]:
# Weighted average (attention-like)
def weighted_average(embeddings, weights):
    """Compute weighted average of embeddings."""
    weights = weights / weights.sum()  # Normalize
    weighted = embeddings * weights.unsqueeze(1)
    return weighted.sum(dim=0)

# Higher weights for important words
weights = torch.tensor([0.1, 0.2, 0.5, 0.1, 0.1])  # Middle word important
weighted_pool = weighted_average(sentence_embeddings, weights)
print(f"Weighted average shape: {weighted_pool.shape}")

## 7. Key Takeaways

### Word Embedding Methods

| Method | Type | Key Idea |
|--------|------|----------|
| **Word2Vec** | Predictive | Skip-gram / CBOW |
| **GloVe** | Count-based | Global co-occurrence |
| **FastText** | Subword | Character n-grams |
| **BERT** | Contextual | Transformer encoder |

### Best Practices

1. **Small datasets**: Use pre-trained embeddings
2. **Domain-specific**: Fine-tune or train from scratch
3. **OOV words**: FastText handles unseen words
4. **Context matters**: Use BERT for polysemy

### Next Steps
- Text classification with embeddings
- Hugging Face transformers for contextual embeddings